# Saffman double-ARP CZ gate：exact backend 验证

这个 notebook 用仓库的 `backend="exact"` 做两原子 state-vector simulation，复现 Saffman 等人在 arXiv:1912.02977 中的 symmetric Rydberg CZ double-ARP protocol。

本地能级写成 `|0>`, `|1>`, `|r>`：`|0>` 是 dark qubit state，pulse 只驱动 `|1> <-> |r>`。也就是说 active transition 正是 passage 里的 `1-r` ARP；保留 `|0>` 只是为了能计算 CZ/Bell fidelity。

Hamiltonian convention 取

$$
H(t)=\sum_{i=1}^2 \left[rac{\Omega(t)}{2}\left(|r_iangle\langle 1_i|+|1_iangle\langle r_i|ight)+\Delta(t)n_i^right]
+B n_1^r n_2^r .
$$

`Omega(t)` 是两个四次 super-Gaussian bumps；每一半 ARP 的 detuning 都从 `-Delta_max` 扫到 `+Delta_max`，中点 `T/2` 在 `Omega=0` 处把 detuning reset 回负值。

本 notebook 按 `scripts/notebooks/run_quench_benchmark.ipynb` 的风格写成顺序脚本：先定义参数和函数型 `DigitalAnalogProtocol`，再跑 exact baseline、画 population、画 fidelity sensitivity。


In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import ryd_gate as rg
from ryd_gate import InteractionSpec
from ryd_gate.lattice import make_chain
from ryd_gate.protocols.digital_analog import DigitalAnalogProtocol

plt.rcParams.update({"figure.dpi": 120})


## 1. Pulse 和系统参数

角频率统一用 rad/s；图上的频率单位再除以 `2*pi` 转成 MHz。`sigma = 0.175 T` 按论文 Fig. 2 caption；每个 ARP pulse 长度为 `T/2`。

`B_POP` 用来画 Fig. 2 风格的 population cycle；`B_GATE` 用来做高 blockade Bell fidelity。这里用 `make_chain(2, spacing_um=1)` 且设置 `C6=B`，所以两原子相互作用正好是 `B=C6/R^6`。


In [ ]:
pi = np.pi
MHz = 2.0 * pi * 1e6
us = 1e-6

Omega_max = 17.0 * MHz
Delta_max = 23.0 * MHz
T_gate = 0.54 * us
T_pulse = 0.5 * T_gate
sigma = 0.175 * T_gate

B_POP = 100.0 * MHz       # Fig. 2 population example
B_GATE = 3000.0 * MHz     # high-fidelity Bell example in the paper text
N_STEPS = 320             # exact backend midpoint piecewise-constant steps
N_SCAN = 13

geom = make_chain(2, spacing_um=1.0)


## 2. Double ARP protocol

这里使用函数型 `DigitalAnalogProtocol`，和 `run_quench_benchmark.ipynb` 里的 `SweepProtocol` 风格一致：continuous double-ARP 波形由 `omega_R_fn(t)` 和 `delta_R_fn(t)` 直接给出，exact backend 再按 `n_steps` 做分段常数演化。

`DigitalAnalogProtocol` 的约定是 Hamiltonian 中有 `-delta_R n_r`，所以为了实现上面写的 `+Delta(t)n_r`，这里传入 `delta_R_fn=lambda t: -arp_delta(t, ...)`。


In [ ]:
arp_offset_a = np.exp(-((T_pulse / 2.0) ** 4) / sigma**4)

def local_arp_time(t):
    t = float(np.clip(t, 0.0, T_gate))
    return t if t < T_pulse else t - T_pulse

def arp_omega(t, omega_scale=1.0):
    u = local_arp_time(t)
    env = (np.exp(-((u - T_pulse / 2.0) ** 4) / sigma**4) - arp_offset_a) / (1.0 - arp_offset_a)
    return float(omega_scale) * Omega_max * env

def arp_delta(t, delta_offset=0.0):
    u = local_arp_time(t)
    return Delta_max * np.sin(pi * (u / T_pulse - 0.5)) + float(delta_offset)

def make_arp_protocol(omega_scale=1.0, delta_offset=0.0, n_steps=N_STEPS):
    return DigitalAnalogProtocol(
        t_gate=T_gate,
        omega_R_fn=lambda t: arp_omega(t, omega_scale=omega_scale),
        delta_R_fn=lambda t: -arp_delta(t, delta_offset=delta_offset),
        omega_hf_fn=lambda t: 0.0,
        delta_hf_fn=lambda t: 0.0,
        n_steps=int(n_steps),
    )

def make_arp_system(B=B_GATE, omega_scale=1.0, delta_offset=0.0, n_steps=N_STEPS):
    protocol = make_arp_protocol(
        omega_scale=omega_scale,
        delta_offset=delta_offset,
        n_steps=n_steps,
    )
    return rg.RydbergSystem.from_lattice(
        geom,
        "01r",
        interaction=InteractionSpec(C6=float(B), mode="all"),
        protocol=protocol,
    )

base_system = make_arp_system(B=B_GATE)
base_protocol = base_system.protocol


## 3. Computational basis 和 Bell benchmark

Hadamard 只作用在 qubit subspace `|0>, |1>`，对 `|r>` 保持不变。Bell benchmark 使用论文里的流程：从 `|11>` 出发，先对两个 qubit 做 Hadamard，再施加 Rydberg CZ operation，最后对 target qubit 做 Hadamard。理想输出是

$$
|B\rangle=(|00\rangle+|11\rangle)/\sqrt2.
$$


In [ ]:
I1 = np.eye(3, dtype=complex)
H_qubit = (1.0 / np.sqrt(2.0)) * np.array(
    [[1.0, 1.0, 0.0],
     [1.0, -1.0, 0.0],
     [0.0, 0.0, np.sqrt(2.0)]],
    dtype=complex,
)
H_both = np.kron(H_qubit, H_qubit)
H_target = np.kron(I1, H_qubit)

ket00 = base_system.product_state(["0", "0"])
ket01 = base_system.product_state(["0", "1"])
ket10 = base_system.product_state(["1", "0"])
ket11 = base_system.product_state(["1", "1"])
ket0r = base_system.product_state(["0", "r"])
ketr0 = base_system.product_state(["r", "0"])
ket1r = base_system.product_state(["1", "r"])
ketr1 = base_system.product_state(["r", "1"])
ketrr = base_system.product_state(["r", "r"])
ket_dplus = (ket1r + ketr1) / np.sqrt(2.0)
ket_bell = (ket00 + ket11) / np.sqrt(2.0)

comp_states = [ket00, ket01, ket10, ket11]
comp_labels = ["00", "01", "10", "11"]

def run_exact_arp(psi0, B=B_GATE, omega_scale=1.0, delta_offset=0.0, t_eval=None, n_steps=N_STEPS):
    system = make_arp_system(
        B=B,
        omega_scale=omega_scale,
        delta_offset=delta_offset,
        n_steps=n_steps,
    )
    return rg.simulate(
        system,
        [],
        psi0,
        backend="exact",
        t_eval=t_eval,
        backend_options={"n_steps": int(n_steps)},
    )

def bell_fidelity(B=B_GATE, omega_scale=1.0, delta_offset=0.0, n_steps=N_STEPS):
    psi_in = H_both @ ket11
    res = run_exact_arp(
        psi_in,
        B=B,
        omega_scale=omega_scale,
        delta_offset=delta_offset,
        n_steps=n_steps,
    )
    psi_out = H_target @ res.psi_final
    return float(abs(np.vdot(ket_bell, psi_out)) ** 2)


## 4. Pulse shape

蓝色是两个 `Omega(t)` bumps；红色是两段 `-Delta_max -> +Delta_max` sweep。中点 reset 发生在 `Omega=0`，所以不会在开耦合时突然混合裸态。


In [ ]:
t_plot = np.linspace(0.0, T_gate, 1201)
omega_plot = np.asarray([arp_omega(t) / MHz for t in t_plot])
delta_plot = np.asarray([arp_delta(t) / MHz for t in t_plot])

fig, ax = plt.subplots(figsize=(8.0, 3.2))
ax.plot(t_plot / us, omega_plot, lw=2.0, label=r"$\Omega(t)/2\pi$")
ax.plot(t_plot / us, delta_plot, lw=2.0, label=r"$\Delta(t)/2\pi$")
ax.axvline(0.5 * T_gate / us, color="k", ls=":", lw=1.0)
ax.set_xlabel("time (us)")
ax.set_ylabel("MHz")
ax.set_title("double ARP pulse")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 5. Population cycle：Fig. 2 的物理图像

这里用 `B_POP/2pi = 100 MHz`，对应论文 Fig. 2 的 population illustration。`|10>` 分支走 `|10> -> |r0> -> |10>`；`|11>` 分支主要走 `|11> -> (|1r>+|r1>)/sqrt(2) -> |11>`，并且有小的 `|rr>` leakage。


In [ ]:
t_eval_pop = np.linspace(0.0, T_gate, N_STEPS + 1)

_t0 = time.perf_counter()
res_10 = run_exact_arp(ket10, B=B_POP, t_eval=t_eval_pop, n_steps=N_STEPS)
res_11 = run_exact_arp(ket11, B=B_POP, t_eval=t_eval_pop, n_steps=N_STEPS)
print(f"population traces elapsed: {time.perf_counter() - _t0:.2f} s")

states_10 = np.asarray(res_10.states)
states_11 = np.asarray(res_11.states)

P_10 = np.abs(states_10 @ ket10.conj()) ** 2
P_r0 = np.abs(states_10 @ ketr0.conj()) ** 2
P_dark_10 = 1.0 - P_10 - P_r0

P_11 = np.abs(states_11 @ ket11.conj()) ** 2
P_dplus = np.abs(states_11 @ ket_dplus.conj()) ** 2
P_rr = np.abs(states_11 @ ketrr.conj()) ** 2
P_dark_11 = 1.0 - P_11 - P_dplus - P_rr

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.4), sharex=True, sharey=True)
axes[0].plot(res_10.times / us, P_10, label=r"$P_{10}$")
axes[0].plot(res_10.times / us, P_r0, label=r"$P_{r0}$")
axes[0].plot(res_10.times / us, P_dark_10, label=r"leak / other")
axes[0].set_title(r"initial $|10\rangle$")

axes[1].plot(res_11.times / us, P_11, label=r"$P_{11}$")
axes[1].plot(res_11.times / us, P_dplus, label=r"$P_{d_+}$")
axes[1].plot(res_11.times / us, P_rr, label=r"$P_{rr}$")
axes[1].plot(res_11.times / us, P_dark_11, label=r"leak / other")
axes[1].set_title(r"initial $|11\rangle$")

for ax in axes:
    ax.set_xlabel("time (us)")
    ax.set_ylabel("population")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.suptitle(rf"double ARP population cycle, $B/2\pi={B_POP / MHz:.0f}$ MHz")
fig.tight_layout()
plt.show()


## 6. Gate phases 和 Bell fidelity

下面用 `B_GATE/2pi = 3 GHz` 打印计算基返回概率、相位和 Bell fidelity。这个 exact backend run 是 coherent upper-bound：没有 Rydberg decay、intermediate-state scattering、Doppler sampling 或 Stark shift compensation。


In [ ]:
_t0 = time.perf_counter()
U_eff = np.zeros((4, 4), dtype=complex)
for col, psi0 in enumerate(comp_states):
    psi_out = run_exact_arp(psi0, B=B_GATE, n_steps=N_STEPS).psi_final
    for row, bra in enumerate(comp_states):
        U_eff[row, col] = np.vdot(bra, psi_out)

phase_diag = np.angle(np.diag(U_eff))
diag_prob = np.abs(np.diag(U_eff)) ** 2
comp_return_prob = np.sum(np.abs(U_eff) ** 2, axis=0)
phi_ent = np.angle(np.exp(1j * (phase_diag[3] - phase_diag[1] - phase_diag[2] + phase_diag[0])))
F0 = bell_fidelity(B=B_GATE, n_steps=N_STEPS)

print(f"gate baseline elapsed: {time.perf_counter() - _t0:.2f} s")
print(f"B/2pi = {B_GATE / MHz:.0f} MHz, Bell fidelity = {F0:.8f}, infidelity = {1.0 - F0:.3e}")
print("state   diag_prob      comp_return      phase/pi")
for label, p_diag, p_comp, ph in zip(comp_labels, diag_prob, comp_return_prob, phase_diag / pi):
    print(f"|{label}>   {p_diag:10.8f}    {p_comp:10.8f}    {ph:10.6f}")
print(f"entangling phase/pi = {phi_ent / pi:.6f}")
print("\nEffective computational block:")
print(np.round(U_eff, 4))


## 7. Fidelity sensitivity：`Omega` 和 detuning

第一张图扫描 Rabi 幅度 `Omega -> s Omega`，范围是 `±5%`。第二张图扫描静态 detuning offset，范围是 `±200 kHz`。这对应论文中用来展示 ARP robustness 的两个误差轴。


In [ ]:
omega_scales = np.linspace(0.95, 1.05, N_SCAN)
delta_offsets_hz = np.linspace(-0.2e6, 0.2e6, N_SCAN)

_t0 = time.perf_counter()
F_omega = np.asarray([
    bell_fidelity(B=B_GATE, omega_scale=float(scale), delta_offset=0.0, n_steps=N_STEPS)
    for scale in omega_scales
])
F_delta = np.asarray([
    bell_fidelity(B=B_GATE, omega_scale=1.0, delta_offset=2.0 * pi * float(offset_hz), n_steps=N_STEPS)
    for offset_hz in delta_offsets_hz
])
print(f"sensitivity scan elapsed: {time.perf_counter() - _t0:.2f} s")
print(f"Omega scan: min F = {F_omega.min():.8f}, max loss from center = {F_omega[len(F_omega)//2] - F_omega.min():.3e}")
print(f"Delta scan: min F = {F_delta.min():.8f}, max loss from center = {F_delta[len(F_delta)//2] - F_delta.min():.3e}")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.4))
axes[0].plot((omega_scales - 1.0) * 100.0, F_omega, "o-", lw=1.8)
axes[0].axvline(0.0, color="k", ls=":", lw=1.0)
axes[0].set_xlabel(r"$\Omega$ variation (%)")
axes[0].set_ylabel("Bell fidelity")
axes[0].set_title(r"Rabi amplitude robustness")
axes[0].grid(alpha=0.25)

axes[1].plot(delta_offsets_hz / 1e3, F_delta, "s-", lw=1.8, color="tab:red")
axes[1].axvline(0.0, color="k", ls=":", lw=1.0)
axes[1].set_xlabel(r"detuning offset / $2\pi$ (kHz)")
axes[1].set_ylabel("Bell fidelity")
axes[1].set_title(r"detuning robustness")
axes[1].grid(alpha=0.25)

fig.suptitle(rf"double ARP CZ sensitivity, $B/2\pi={B_GATE / MHz:.0f}$ MHz")
fig.tight_layout()
plt.show()


## 8. 读图方式

- `B_POP = 100 MHz` 的 population plot 用来恢复 Fig. 2 的直观图像。
- `B_GATE = 3 GHz` 的 fidelity plot 用来检查高 blockade 下 Bell fidelity 对 `Omega` 和 detuning offset 的鲁棒性。
- 如果把 `B_GATE` 改成 `100 MHz`，`|rr>` leakage 会明显增大，Bell fidelity 会下降；这不是 pulse 形状错误，而是 blockade 不够强。
- 这里没有加入 Rydberg decay、intermediate-state scattering、Doppler sampling 或 Stark shift compensation；这些会把 coherent upper-bound fidelity 降低到更接近论文的 full error model。


## 9. `our` seven-level system：同一个 double ARP protocol

上面 `01r/01r-like` 部分只保留了 dark `|0>`、computational `|1>` 和 Rydberg `|r>`。下面切到 `scripts/notebooks/cz_gate_validation_and_errors.ipynb` 使用的 `our` 七能级系统：

```python
RydbergSystem.from_lattice(make_chain(2, spacing_um=3.0), "rb87_7", param_set="our", blackmanflag=True, detuning_sign=1)
```

这里使用 `src/ryd_gate/protocols/gate_cz_double_arp.py` 里的 `DoubleARPProtocol`。它输出 `rb87_7` 已有的 `drive_420`、`drive_420_dag` 和 `lightshift_zero` channels：

- effective `Omega(t)` 映射成 420nm amplitude scale，比例为 `Omega(t) / system.meta("rabi_eff")`；
- effective `Delta(t)` 映射成 420nm optical phase 的时间导数；
- `compensate_stark=True` 时，从 `our` 本地矩阵块估算 420 动态 differential Stark shift，并把它加入 chirp；
- `theta` 不再固定为 `pi`，而是先由 coherent overlaps 重新拟合。

注意 `our` 系统的 `rabi_eff/2pi` 约为 5 MHz；若直接套用 Saffman 的 `Omega_max/2pi=17 MHz`，420nm amplitude scale 会大于 1。这是一个强驱动映射测试，不等同于已经为 `our` 参数重新优化过的 ARP gate。


In [ ]:
from IPython.display import display, Markdown

from ryd_gate.protocols.gate_cz_double_arp import DoubleARPProtocol
from ryd_gate.analysis.gate_metrics import average_gate_infidelity, error_budget, population_evolution
from ryd_gate.backends.exact import simulate as simulate_exact
from ryd_gate.core.operators import build_occ_operator

OUR_SPACING_UM = 3.0
OUR_N_STEPS = 80
OUR_OMEGA_MAX = Omega_max
OUR_DELTA_MAX = Delta_max
OUR_T_GATE = T_gate
OUR_SIGMA = sigma
OUR_COMPENSATE_STARK = True
OUR_STARK_COMPENSATION_SIGN = -1.0


def make_our_arp_protocol(theta=np.pi):
    return DoubleARPProtocol(
        omega_max=OUR_OMEGA_MAX,
        delta_max=OUR_DELTA_MAX,
        t_gate=OUR_T_GATE,
        sigma=OUR_SIGMA,
        theta=float(theta),
        n_steps=OUR_N_STEPS,
        compensate_stark=OUR_COMPENSATE_STARK,
        stark_compensation_sign=OUR_STARK_COMPENSATION_SIGN,
    )


def make_our_arp_system(protocol=None, **flags):
    return rg.RydbergSystem.from_lattice(
        make_chain(2, spacing_um=OUR_SPACING_UM),
        "rb87_7",
        param_set="our",
        blackmanflag=True,
        detuning_sign=1,
        protocol=make_our_arp_protocol() if protocol is None else protocol,
        **flags,
    )


our_arp_protocol = make_our_arp_protocol()
our_system = make_our_arp_system(protocol=our_arp_protocol)
our_params = our_arp_protocol.unpack_params([], our_system)
print(f"our rb87_7 dim = {our_system.dim}")
print(f"rabi_eff/2pi = {our_system.meta('rabi_eff') / MHz:.3f} MHz")
print(f"ARP Omega_max/rabi_eff = {OUR_OMEGA_MAX / our_system.meta('rabi_eff'):.3f}")
print(f"V_ryd/2pi at {OUR_SPACING_UM} um = {our_system.meta('v_ryd') / MHz:.3f} MHz")
print(f"Stark compensation enabled = {OUR_COMPENSATE_STARK}, sign = {OUR_STARK_COMPENSATION_SIGN:+.0f}")

_t0 = time.perf_counter()
comp_basis = {label: our_system.product_state(label) for label in ("00", "01", "10", "11")}
phase_basis = {label: comp_basis[label] for label in ("00", "01", "11")}
phase_results = {}
phase_overlaps = {}
phase_comp_return = {}
phase_residuals = {"e1": 0.0, "e2": 0.0, "e3": 0.0, "ryd": 0.0, "ryd_garb": 0.0}
occ_ops = {
    "e1": build_occ_operator(2),
    "e2": build_occ_operator(3),
    "e3": build_occ_operator(4),
    "ryd": build_occ_operator(5),
    "ryd_garb": build_occ_operator(6),
}

for label, psi0 in phase_basis.items():
    res = simulate_exact(our_system, [], psi0)
    psi_f = res.psi_final
    phase_results[label] = psi_f
    phase_overlaps[label] = np.vdot(psi0, psi_f)
    phase_comp_return[label] = sum(abs(np.vdot(comp_basis[other], psi_f)) ** 2 for other in comp_basis)
    for key, op in occ_ops.items():
        phase_residuals[key] += float(np.real(np.vdot(psi_f, op @ psi_f))) / len(phase_basis)


def avg_fidelity_for_theta(theta):
    a00 = phase_overlaps["00"]
    a01 = np.exp(-1.0j * theta) * phase_overlaps["01"]
    a11 = np.exp(-1.0j * (2.0 * theta + np.pi)) * phase_overlaps["11"]
    return float((abs(a00 + 2.0 * a01 + a11) ** 2 + abs(a00) ** 2 + 2.0 * abs(a01) ** 2 + abs(a11) ** 2) / 20.0)

theta_grid = np.linspace(-np.pi, np.pi, 4001)
fid_grid = np.asarray([avg_fidelity_for_theta(theta) for theta in theta_grid])
theta_seed = float(theta_grid[int(np.argmax(fid_grid))])
theta_refine = np.linspace(theta_seed - 0.01, theta_seed + 0.01, 2001)
fid_refine = np.asarray([avg_fidelity_for_theta(theta) for theta in theta_refine])
theta_opt = float(theta_refine[int(np.argmax(fid_refine))])
our_avg_inf = 1.0 - float(fid_refine.max())
our_avg_inf_theta_pi = 1.0 - avg_fidelity_for_theta(np.pi)

our_arp_protocol = make_our_arp_protocol(theta=theta_opt)
our_system = make_our_arp_system(protocol=our_arp_protocol)

print(f"our coherent average infidelity at theta=pi  = {our_avg_inf_theta_pi:.6e}")
print(f"optimized theta = {theta_opt:.6f} rad ({theta_opt / np.pi:.6f} pi)")
print(f"our coherent average infidelity optimized = {our_avg_inf:.6e}; elapsed = {time.perf_counter() - _t0:.2f} s")
print("return probabilities:", {k: float(abs(v) ** 2) for k, v in phase_overlaps.items()})
print("computational return:", {k: float(v) for k, v in phase_comp_return.items()})
print("final residual populations:", {k: float(v) for k, v in phase_residuals.items()})

pops = population_evolution(our_system, our_arp_protocol, [], "SSS-0")
time_ns = np.asarray(pops["t_list"]) * 1e9

fig, ax = plt.subplots(figsize=(8.0, 4.2))
ax.plot(time_ns, (pops["e1"] + pops["e2"] + pops["e3"]) / 2.0, lw=1.6, label="intermediate")
ax.plot(time_ns, pops["ryd"] / 2.0, lw=1.6, label="Rydberg |r1>")
ax.plot(time_ns, pops["ryd_garb"] / 2.0, lw=1.6, label="garbage Rydberg |r2>")
ax.set_xlabel("time (ns)")
ax.set_ylabel("SSS-0 averaged population per atom")
ax.set_title("our rb87_7 Stark-compensated double-ARP population evolution")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## 10. 为什么 `our` 结果不是相位门：动态 Stark shift 诊断

Saffman 的 double-ARP 参数是给有效二能级 Hamiltonian 写的。映射到 `our rb87_7` 后，`Omega_max/2pi=17 MHz` 需要把 420nm amplitude scale 放大到 `Omega_max/rabi_eff≈3.4`。这会产生很大的 `|1>` AC Stark shift。

下面用 `our_system` 的本地矩阵块做二阶消元估算。若 differential Stark shift 大于 detuning sweep range，ARP sweep 就没有穿过真实 avoided crossing，末态不会回到计算基，因此不是相位门。


In [ ]:
H0_local = our_system.blocks.get("H_const").matrix
H420_local = our_system.blocks.get("drive_420").matrix
H1013_local = our_system.blocks.get("H_1013").matrix
mid_energies = np.real(np.diag(H0_local)[2:5])
amp_peak = OUR_OMEGA_MAX / our_system.meta("rabi_eff")

stark_1_peak = -sum(abs(amp_peak * H420_local[e, 1]) ** 2 / mid_energies[e - 2] for e in (2, 3, 4))
stark_r_peak = -sum(abs(H1013_local[5, e]) ** 2 / mid_energies[e - 2] for e in (2, 3, 4))
g_eff_peak = -sum(H1013_local[5, e] * amp_peak * H420_local[e, 1] / mid_energies[e - 2] for e in (2, 3, 4))
g_garb_peak = -sum(H1013_local[6, e] * amp_peak * H420_local[e, 1] / mid_energies[e - 2] for e in (2, 3, 4))
differential_stark = stark_r_peak - stark_1_peak

print(f"effective Omega_peak/2pi from local blocks = {2 * abs(g_eff_peak) / MHz:.3f} MHz")
print(f"garbage Rydberg coupling/2pi         = {2 * abs(g_garb_peak) / MHz:.3f} MHz")
print(f"|1> Stark shift /2pi                 = {stark_1_peak / MHz:.3f} MHz")
print(f"|r> Stark shift /2pi                 = {stark_r_peak / MHz:.3f} MHz")
print(f"differential Stark (r-1)/2pi          = {differential_stark / MHz:.3f} MHz")
print(f"ARP desired detuning at peak /2pi     = {our_arp_protocol.detuning(T_gate / 4, our_params) / MHz:.3f} MHz")
print(f"compensated chirp at peak /2pi        = {our_arp_protocol.chirp_detuning(T_gate / 4, our_params) / MHz:.3f} MHz")
print(f"chirp + Stark at peak /2pi            = {(our_arp_protocol.chirp_detuning(T_gate / 4, our_params) + our_arp_protocol.stark_shift(T_gate / 4, our_params)) / MHz:.3f} MHz")
print(f"ARP detuning sweep range /2pi         = ±{OUR_DELTA_MAX / MHz:.3f} MHz")


In [ ]:
# Deterministic noise diagnose, mirroring cz_gate_validation_and_errors.ipynb.
# This is intentionally a compact table: no Monte Carlo dephasing/position sampling here.

RUN_OUR_NOISE_DIAGNOSE = False
OUR_BUDGET_STATES = ["01", "11"]

if RUN_OUR_NOISE_DIAGNOSE and our_avg_inf < 1e-2:
    noise_rows = []

    _t0 = time.perf_counter()
    system_ryd = make_our_arp_system(enable_rydberg_decay=True)
    inf_ryd = average_gate_infidelity(system_ryd, our_arp_protocol, [])
    budget_ryd = error_budget(system_ryd, our_arp_protocol, [], initial_states=OUR_BUDGET_STATES)["rydberg_decay"]
    noise_rows.append(("rydberg_decay", inf_ryd, budget_ryd))

    system_mid = make_our_arp_system(enable_intermediate_decay=True)
    inf_mid = average_gate_infidelity(system_mid, our_arp_protocol, [])
    budget_mid = error_budget(system_mid, our_arp_protocol, [], initial_states=OUR_BUDGET_STATES)["intermediate_decay"]
    noise_rows.append(("intermediate_decay", inf_mid, budget_mid))

    system_mid_no0 = make_our_arp_system(enable_intermediate_decay=True, enable_0_scattering=False)
    inf_mid_no0 = average_gate_infidelity(system_mid_no0, our_arp_protocol, [])
    zero_scatter_extra = max(0.0, float(inf_mid - inf_mid_no0))

    system_all = make_our_arp_system(
        enable_rydberg_decay=True,
        enable_intermediate_decay=True,
        enable_polarization_leakage=True,
    )
    inf_all = average_gate_infidelity(system_all, our_arp_protocol, [])
    budget_all = error_budget(system_all, our_arp_protocol, [], initial_states=OUR_BUDGET_STATES)
    combined = {
        "XYZ": budget_all["rydberg_decay"]["XYZ"] + budget_all["intermediate_decay"]["XYZ"] + budget_all["polarization_leakage"]["XYZ"],
        "AL": budget_all["rydberg_decay"]["AL"] + budget_all["intermediate_decay"]["AL"] + budget_all["polarization_leakage"]["AL"],
        "LG": budget_all["rydberg_decay"]["LG"] + budget_all["intermediate_decay"]["LG"] + budget_all["polarization_leakage"]["LG"],
    }
    noise_rows.append(("all_deterministic", inf_all, combined))

    text = "| source | average infidelity | XYZ | AL | LG | residual |\n|---|---:|---:|---:|---:|---:|\n"
    for label, inf, budget in noise_rows:
        xyz = float(budget.get("XYZ", 0.0))
        al = float(budget.get("AL", 0.0))
        lg = float(budget.get("LG", 0.0))
        residual = float(inf) - (xyz + al + lg)
        if label == "intermediate_decay":
            residual -= zero_scatter_extra
        text += f"| {label} | {float(inf):.6e} | {xyz:.6e} | {al:.6e} | {lg:.6e} | {residual:.6e} |\n"
    text += f"\nIntermediate `|0>` scattering extra infidelity: `{zero_scatter_extra:.6e}`.  \n"
    text += f"Noise diagnose elapsed: `{time.perf_counter() - _t0:.2f} s`."
    display(Markdown(text))
else:
    print("Noise diagnose skipped: first retune the coherent ARP gate so it is actually a phase gate. Set RUN_OUR_NOISE_DIAGNOSE=True after our_avg_inf < 1e-2.")
